In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange
import concurrent.futures
import unittest
from concurrent.futures import ThreadPoolExecutor
import PIL.Image
import cairosvg
from io import BytesIO
import uuid
import time

sys.path.append("../../")
import biked_commons
from biked_commons.api.rendering import RenderingEngine
from biked_commons.bike_embedding.clip_embedding_calculator import ClipEmbeddingCalculator
from biked_commons.resource_utils import resource_path


Using java as the Java binary


In [2]:
data = pd.read_csv("../../resources/datasets/split_datasets/bike_bench.csv", index_col=0)
data = data.iloc[:60]

In [3]:
CHECKPOINT_SIZE = 100
def generate_embeddings(data, save_dir, number_rendering_servers: int, server_init_timeout_seconds: int = 90, ):
    data_ids = data.index.tolist()
    records = data.to_dict(orient="records")
    executor = ThreadPoolExecutor(max_workers=number_rendering_servers)
    rendering_engine = RenderingEngine(number_rendering_servers=number_rendering_servers,
                                       server_init_timeout_seconds=server_init_timeout_seconds)
    
    start_time = time.time()

    # embedding_calculator = ClipEmbeddingCalculator()

    def render_record(inputs: list):
        
        clip_record, data_id = inputs
        rendering_result = rendering_engine.render_clip(clip_record)
        png_data = cairosvg.svg2png(rendering_result.image_bytes)
        xml_data = rendering_result.xml_file
        # random_id = uuid.uuid4()
        id_path = os.path.join(save_dir, str(data_id))
        with open(f"{id_path}.bcad", "w") as f:
            f.write(xml_data)
        with open(f"{id_path}.png", "wb") as f:
            f.write(png_data)
        print(f"Rendering {data_id} completed at time {time.time() - start_time} seconds")

        # image = PIL.Image.open(BytesIO(png_data))
        # print("Image loaded...")
        # image = torch.tensor(np.array(image))
        # embedding_tensor = embedding_calculator.embed_images(image)
        # print("Embedding tensor obtained...") 
        
    for i in range(len(records)):
    # for record in records:
        data_id = data_ids[i]
        record = records[i]
        executor.submit(render_record, [record, data_id])
    executor.shutdown(wait=True)
    finish_time = time.time()
    print(f"Total time taken: {finish_time - start_time} seconds")


In [4]:
save_dir = resource_path("bike_bench_rendering")
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
generate_embeddings(data, save_dir, number_rendering_servers=10, server_init_timeout_seconds=180)

Starting BikeCAD server on port 8082...Starting BikeCAD server on port 8084...
Starting BikeCAD server on port 8083...
Starting BikeCAD server on port 8081...

Starting BikeCAD server on port 8085...
Starting BikeCAD server on port 8086...
Starting BikeCAD server on port 8087...
Starting BikeCAD server on port 8089...
Starting BikeCAD server on port 8088...
BikeCAD server started on port 8081.
BikeCAD server started on port 8088.
BikeCAD server started on port 8085.
BikeCAD server started on port 8087.
BikeCAD server started on port 8086.
BikeCAD server started on port 8083.
BikeCAD server started on port 8084.
BikeCAD server started on port 8082.
BikeCAD server started on port 8089.
http://localhost:8080/api/v1/render
http://localhost:8081/api/v1/render
http://localhost:8082/api/v1/render
http://localhost:8083/api/v1/render
http://localhost:8084/api/v1/render
http://localhost:8085/api/v1/render
http://localhost:8086/api/v1/render
http://localhost:8087/api/v1/render
http://localhost:80